In [ ]:
# P3. 지출 예측

목적: 고객별 **이번 달 말 예상 지출**, **다음 달 대분류별 지출 예측**, **백테스트 및 MAE 비교**를 수행한다.

설계 기준:
- 이번 달 말 예상: 현재까지 지출 ÷ 경과일수 × 해당 월 일수
- 다음 달 예측 기본 모델: 최근 3개월 이동평균
- 비교 모델: 최근 3개월 선형 추세
- 1~11월 데이터를 이용해 다음 달을 예측하고 실제값과 비교
- MAE가 낮은 모델을 확인

`src/forecast.py`의 함수와 동일한 데이터를 사용해 Pandas 결과를 검증한다.
# 1. 라이브러리 및 프로젝트 경로 설정
from pathlib import Path
import sys
import calendar

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()

# VS Code에서 notebooks 폴더를 기준으로 실행하는 경우 프로젝트 루트로 이동
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_PATH = PROJECT_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

print("PROJECT_ROOT:", PROJECT_ROOT)

# 2. 데이터 불러오기
# 기본 프로젝트 구조: data/transactions.csv

DATA_PATH = PROJECT_ROOT / "data" / "transactions.csv"

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"transactions.csv를 찾을 수 없습니다: {DATA_PATH}\n"
        "프로젝트 구조가 repo/data/transactions.csv인지 확인하세요."
    )

df = pd.read_csv(DATA_PATH)

print("행 수:", len(df))
print("컬럼:", df.columns.tolist())
display(df.head())

# 3. P3에서 사용할 공통 컬럼 정리
# P1 loader가 이미 dt/ym을 만들어 주는 경우 그대로 사용한다.
# 아직 loader를 사용하지 않는 경우를 대비해 필요한 컬럼을 생성한다.

work = df.copy()

if "dt" not in work.columns:
    if "transaction_datetime" not in work.columns:
        raise ValueError("dt 또는 transaction_datetime 컬럼이 필요합니다.")
    work["dt"] = pd.to_datetime(work["transaction_datetime"], errors="coerce")
else:
    work["dt"] = pd.to_datetime(work["dt"], errors="coerce")

work = work.dropna(subset=["dt"])

if "ym" not in work.columns:
    work["ym"] = work["dt"].dt.strftime("%Y-%m")

work["amount"] = pd.to_numeric(work["amount"], errors="coerce").fillna(0)

if "big_category" not in work.columns:
    if "category" not in work.columns:
        raise ValueError("big_category 또는 category 컬럼이 필요합니다.")
    work["big_category"] = work["category"].astype(str).str.strip().str.upper()

print(work[["customer_id", "dt", "ym", "amount", "big_category"]].head())

## 4. 데이터 기본 확인

P3 예측을 실행하기 전에 날짜 범위, 고객 수, 월별 데이터가 정상인지 확인한다.
print("데이터 기간:", work["dt"].min(), "~", work["dt"].max())
print("고객 수:", work["customer_id"].nunique())
print("월 수:", work["ym"].nunique())

monthly_check = (
    work.groupby("ym", as_index=False)["amount"]
    .sum()
    .sort_values("ym")
)

display(monthly_check)

## 5. 월별·대분류별 지출 집계

다음 달 예측의 기본 단위는 **고객 × 월 × 대분류**다.
monthly_category = (
    work.groupby(
        ["customer_id", "ym", "big_category"],
        as_index=False
    )["amount"]
    .sum()
    .rename(columns={"amount": "monthly_amount"})
    .sort_values(["customer_id", "ym", "big_category"])
)

display(monthly_category.head(20))

## 6. 최근 3개월 이동평균 함수

예측 대상 월의 직전 3개월을 사용한다. 데이터가 부족한 초기 구간은 백테스트에서 제외한다.
def moving_average_predict(history_values, window=3):
    values = pd.Series(history_values).dropna().tail(window)
    if len(values) < window:
        return np.nan
    return float(values.mean())


def linear_trend_predict(history_values, window=3):
    values = pd.Series(history_values).dropna().tail(window).to_numpy(dtype=float)
    if len(values) < window:
        return np.nan

    x = np.arange(len(values))
    slope, intercept = np.polyfit(x, values, 1)
    prediction = slope * len(values) + intercept
    return max(0.0, float(prediction))

## 7. 대분류별 다음 달 예측

특정 고객의 기준 월을 넣으면 다음 달의 대분류별 이동평균/선형추세 예측값을 만든다.
def forecast_next_month_pandas(customer_df, base_month):
    base_date = pd.to_datetime(base_month + "-01")
    next_month = (base_date + pd.DateOffset(months=1)).strftime("%Y-%m")

    monthly = (
        customer_df.groupby(["ym", "big_category"], as_index=False)["amount"]
        .sum()
        .rename(columns={"amount": "monthly_amount"})
    )

    results = []

    for category, group in monthly.groupby("big_category"):
        group = group.sort_values("ym")
        history = group[group["ym"] < next_month].tail(3)

        ma = moving_average_predict(history["monthly_amount"], 3)
        trend = linear_trend_predict(history["monthly_amount"], 3)

        results.append({
            "big_category": category,
            "moving_average": ma,
            "linear_trend": trend,
        })

    result = pd.DataFrame(results)
    if not result.empty:
        result["selected_forecast"] = result["moving_average"]

    return next_month, result

# 8. 실제 고객 1명으로 다음 달 예측 확인

CUSTOMER_ID = str(work["customer_id"].iloc[0])
BASE_MONTH = sorted(work["ym"].unique())[-1]

customer_df = work[
    work["customer_id"].astype(str) == CUSTOMER_ID
].copy()

next_month, forecast_table = forecast_next_month_pandas(
    customer_df,
    BASE_MONTH
)

print("고객 ID:", CUSTOMER_ID)
print("기준 월:", BASE_MONTH)
print("예측 월:", next_month)

display(forecast_table)

if not forecast_table.empty:
    print("이동평균 전체 예측:", forecast_table["moving_average"].sum())
    print("선형추세 전체 예측:", forecast_table["linear_trend"].sum())

## 9. 이번 달 말 예상 지출

공식:

`현재까지 지출 ÷ 경과 일수 × 해당 월 일수`

고정비는 P5 결과를 연결한 뒤 별도로 더한다. 현재 노트북에서는 변동지출 페이스만 계산한다.
def current_month_forecast_pandas(customer_df, month):
    month_df = customer_df[customer_df["ym"] == month].copy()

    month_start = pd.to_datetime(month + "-01")
    days_in_month = calendar.monthrange(month_start.year, month_start.month)[1]

    if month_df.empty:
        return {
            "actual_to_date": 0.0,
            "elapsed_days": 0,
            "days_in_month": days_in_month,
            "forecast": 0.0,
        }

    actual_to_date = float(month_df["amount"].sum())
    elapsed_days = int(month_df["dt"].max().day)

    forecast = (
        actual_to_date / elapsed_days * days_in_month
        if elapsed_days > 0 else 0.0
    )

    return {
        "actual_to_date": actual_to_date,
        "elapsed_days": elapsed_days,
        "days_in_month": days_in_month,
        "forecast": float(forecast),
    }


current_month_result = current_month_forecast_pandas(
    customer_df,
    BASE_MONTH
)

display(pd.DataFrame([current_month_result]))

## 10. 백테스트

1~11월 데이터를 사용해 다음 달을 예측하고 실제값과 비교한다.

- 1~3월 → 4월
- 2~4월 → 5월
- ...
- 9~11월 → 12월

대분류별 예측값을 합산해 고객의 월 전체 지출을 예측한다.
def backtest_customer(customer_df, min_history=3):
    monthly = (
        customer_df.groupby("ym", as_index=False)["amount"]
        .sum()
        .rename(columns={"amount": "actual_amount"})
        .sort_values("ym")
    )

    months = monthly["ym"].tolist()
    results = []

    for i in range(min_history, len(months)):
        target_month = months[i]
        history = monthly.iloc[i-min_history:i]["actual_amount"]

        ma_pred = moving_average_predict(history, min_history)
        trend_pred = linear_trend_predict(history, min_history)
        actual = float(monthly.iloc[i]["actual_amount"])

        results.append({
            "actual_month": target_month,
            "actual_amount": actual,
            "moving_average": ma_pred,
            "linear_trend": trend_pred,
            "ma_error": abs(ma_pred - actual),
            "trend_error": abs(trend_pred - actual),
        })

    return pd.DataFrame(results)


backtest = backtest_customer(customer_df)
display(backtest)

# 11. MAE 비교

if backtest.empty:
    print("백테스트에 필요한 최소 4개월 데이터가 없습니다.")
else:
    mae_ma = backtest["ma_error"].mean()
    mae_trend = backtest["trend_error"].mean()

    mae_result = pd.DataFrame({
        "model": ["3개월 이동평균", "3개월 선형추세"],
        "MAE": [mae_ma, mae_trend],
    })

    display(mae_result)

    print(f"이동평균 MAE: {mae_ma:,.0f}원")
    print(f"선형추세 MAE: {mae_trend:,.0f}원")
    print("※ 실제 프로젝트에서는 이 결과를 기준으로 최종 모델을 결정한다.")

## 12. 모든 고객 기준 백테스트

특정 고객 한 명이 아니라 전체 고객을 대상으로 모델 성능을 확인한다.
all_backtests = []

for customer_id, group in work.groupby("customer_id"):
    result = backtest_customer(group)

    if result.empty:
        continue

    result["customer_id"] = customer_id
    all_backtests.append(result)

if all_backtests:
    all_backtest_df = pd.concat(all_backtests, ignore_index=True)

    overall_mae = pd.DataFrame({
        "model": ["3개월 이동평균", "3개월 선형추세"],
        "MAE": [
            all_backtest_df["ma_error"].mean(),
            all_backtest_df["trend_error"].mean(),
        ],
    })

    display(overall_mae)
else:
    print("백테스트 가능한 고객이 없습니다.")

## 13. `src/forecast.py` 결과 확인

최종 모듈의 공통 인터페이스인 `analyze(df, customer_id, month)`가 정상적으로 동작하는지 확인한다.
try:
    from forecast import analyze

    result = analyze(
        work,
        customer_id=CUSTOMER_ID,
        month=BASE_MONTH,
    )

    print("feature:", result["feature"])
    print("title:", result["title"])
    print("message:", result["message"])

except Exception as e:
    print("forecast.py 실행 중 오류:", repr(e))

## 14. 검증 체크리스트

- [ ] `transactions.csv` 정상 로드
- [ ] `dt`, `ym`, `amount`, `big_category` 확인
- [ ] 고객 × 월 × 대분류 집계 확인
- [ ] 이번 달 말 예상 지출 계산 확인
- [ ] 최근 3개월 이동평균 계산 확인
- [ ] 선형 추세 계산 확인
- [ ] 1~11월 백테스트 수행
- [ ] MAE 비교
- [ ] `src/forecast.py`의 `analyze()` 실행 확인
- [ ] SQL 결과와 Pandas 핵심 수치 교차검증

최종 발표에서는 실제 고객 1명의 예측 결과와 백테스트 MAE를 근거로 설명한다.